In [1]:
import pandas as pd

df = pd.read_csv("teams_labor.csv")
df2 = pd.read_csv("gt_entries.csv")

new = (
   pd.concat([df["Team"], df2["Code"]], ignore_index=True)
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

MU = 25.0
SIGMA = MU / 3

df_ts = pd.DataFrame(
    {
        "Team": new,
        "Mu": MU,
        "Sigma": SIGMA,
        "Aff_Mu": MU,
        "Aff_Sigma": SIGMA,
        "Neg_Mu": MU,
        "Neg_Sigma": SIGMA,
        "Aff_Rounds": 0,
        "Neg_Rounds": 0,
    }
)

df_ts.to_csv("teams_labor_init.csv", index=False)

In [2]:
def find_duplicates(series):
  duplicates = []
  seen = set()
  for item in series:
    reversed_last_two = item[-2:][::-1]
    prefix = item[:-2]
    key = prefix + reversed_last_two
    if key in seen or item in seen:
      duplicates.append(item)
    else:
      seen.add(item)
      seen.add(key)
  return duplicates

def reverse(series):
    transformed = []
    for s in series:
      transformed.append(s[:-2] + s[-2:][::-1])
    return transformed

teams = pd.read_csv('teams_labor_init.csv')
dupes = find_duplicates(teams['Team'])
dupes2 = reverse(dupes)
df = pd.DataFrame({'1':dupes2, '2':dupes})

for i in range(len(df['1'])):
    print('df[column] = df[column].str.replace(\''+df['2'][i]+'\',\''+df['1'][i]+'\')')

df[column] = df[column].str.replace('Baylor PM','Baylor MP')
df[column] = df[column].str.replace('Harvard SJ','Harvard JS')
df[column] = df[column].str.replace('Kansas LH','Kansas HL')
df[column] = df[column].str.replace('Kansas PB','Kansas BP')
df[column] = df[column].str.replace('Kentucky SR','Kentucky RS')
df[column] = df[column].str.replace('UTD RP','UTD PR')
df[column] = df[column].str.replace('Western Kentucky NK','Western Kentucky KN')


In [3]:
def clean_teams(df, column):
    df[column] = df[column].str.replace(' - ONLINE','')
    df[column] = df[column].str.replace(' - HYBRID','')
    df[column] = df[column].str.replace('Baylor PM','Baylor MP')
    df[column] = df[column].str.replace('Harvard SJ','Harvard JS')
    df[column] = df[column].str.replace('Kansas LH','Kansas HL')
    df[column] = df[column].str.replace('Kentucky SR','Kentucky RS')
    df[column] = df[column].str.replace('UTD RP','UTD PR')
    df[column] = df[column].str.replace('Western Kentucky NK','Western Kentucky KN')
    return df[column]

def clean_rd(rd):
    df = pd.read_csv('data_labor/'+rd+'.csv')
    df['Aff'] = clean_teams(df, 'Aff')
    df['Neg'] = clean_teams(df, 'Neg')
    for index, row in df.iterrows():
        if 'AFF' in row['Win']:
            df.at[index, 'Win'] = 'Aff'
        if 'NEG' in row['Win']:
            df.at[index, 'Win'] = 'Neg'
    df = df[['Aff','Neg','Win']]
    df.to_csv('data_labor/'+rd+'.csv')
    return df[column]

def clean_rd(rd):
    df = pd.read_csv('data_labor/'+rd+'.csv')
    df['Aff'] = clean_teams(df, 'Aff')
    df['Neg'] = clean_teams(df, 'Neg')
    for index, row in df.iterrows():
        if 'AFF' in row['Win']:
            df.at[index, 'Win'] = 'Aff'
        if 'NEG' in row['Win']:
            df.at[index, 'Win'] = 'Neg'
    df = df[['Aff','Neg','Win']]
    df.to_csv('data_labor/'+rd+'.csv')

In [4]:
import pandas as pd
import numpy as np
from trueskill import TrueSkill, Rating, rate_1vs1

def _ensure_ts_columns(teams: pd.DataFrame, env: TrueSkill):
    """
    Ensure teams DataFrame has TrueSkill columns.
    """
    defaults = {
        'Mu': env.mu, 'Sigma': env.sigma,
        'Aff_Mu': env.mu, 'Aff_Sigma': env.sigma,
        'Neg_Mu': env.mu, 'Neg_Sigma': env.sigma,
        'Aff_Rounds': 0, 'Neg_Rounds': 0,
    }
    for col, val in defaults.items():
        if col not in teams.columns:
            teams[col] = val
        else:
            teams[col] = teams[col].fillna(val)
    return teams


def setup(teams: pd.DataFrame, rd: pd.DataFrame, env: TrueSkill | None = None):
    """
    Prepare matchups.
    rd must have columns: ['Aff','Neg','Win']
    """
    env = env or TrueSkill()
    teams = _ensure_ts_columns(teams.copy(), env)

    aff = pd.merge(rd, teams, left_on='Aff', right_on='Team', suffixes=('', '_Aff'))
    neg = pd.merge(rd, teams, left_on='Neg', right_on='Team', suffixes=('', '_Neg'))
    return aff, neg, env, teams


def compute_new_trueskill(aff: pd.DataFrame, neg: pd.DataFrame, env):
    """
    Apply TrueSkill updates row by row.
    NOTE: After merge(), columns like Mu/Sigma/Aff_Mu/Neg_Mu do NOT get suffixed
    because rd has no overlapping columns. So we reference unsuffixed names.
    """
    import numpy as np
    from trueskill import Rating, rate_1vs1

    out_aff, out_neg = [], []

    for i in range(len(aff)):
        a = aff.iloc[i]
        n = neg.iloc[i]

        # Overall ratings
        r_aff_overall = Rating(a['Mu'], a['Sigma'])
        r_neg_overall = Rating(n['Mu'], n['Sigma'])

        # Side-specific ratings
        r_aff_side = Rating(a['Aff_Mu'], a['Aff_Sigma'])
        r_neg_side = Rating(n['Neg_Mu'], n['Neg_Sigma'])

        win = a.get('Win', np.nan)

        if win == 'Aff':
            w_overall, l_overall = rate_1vs1(r_aff_overall, r_neg_overall, env=env)
            w_side, l_side       = rate_1vs1(r_aff_side,    r_neg_side,    env=env)

            new_aff_overall, new_neg_overall = w_overall, l_overall
            new_aff_side,    new_neg_side    = w_side,    l_side
            inc_aff, inc_neg = 1, 1

        elif win == 'Neg':
            w_overall, l_overall = rate_1vs1(r_neg_overall, r_aff_overall, env=env)
            w_side,    l_side    = rate_1vs1(r_neg_side,    r_aff_side,    env=env)

            new_aff_overall, new_neg_overall = l_overall, w_overall
            new_aff_side,    new_neg_side    = l_side,    w_side
            inc_aff, inc_neg = 1, 1

        else:
            # No result recorded -> no update
            new_aff_overall, new_neg_overall = r_aff_overall, r_neg_overall
            new_aff_side,    new_neg_side    = r_aff_side,    r_neg_side
            inc_aff, inc_neg = 0, 0

        out_aff.append({
            'Aff': a['Aff'],
            'New_Mu': new_aff_overall.mu,
            'New_Sigma': new_aff_overall.sigma,
            'New_Aff_Mu': new_aff_side.mu,
            'New_Aff_Sigma': new_aff_side.sigma,
            'Inc_Aff_Rounds': inc_aff
        })
        out_neg.append({
            'Neg': n['Neg'],
            'New_Mu': new_neg_overall.mu,
            'New_Sigma': new_neg_overall.sigma,
            'New_Neg_Mu': new_neg_side.mu,
            'New_Neg_Sigma': new_neg_side.sigma,
            'Inc_Neg_Rounds': inc_neg
        })

    return pd.DataFrame(out_aff), pd.DataFrame(out_neg)

def trueskill_merge(teams: pd.DataFrame, aff_updates: pd.DataFrame, neg_updates: pd.DataFrame):
    """
    Merge updated ratings back into teams DataFrame.
    """
    t = teams.merge(
        aff_updates, left_on='Team', right_on='Aff', how='left'
    )
    t['Mu'] = t['New_Mu'].combine_first(t['Mu'])
    t['Sigma'] = t['New_Sigma'].combine_first(t['Sigma'])
    t['Aff_Mu'] = t['New_Aff_Mu'].combine_first(t['Aff_Mu'])
    t['Aff_Sigma'] = t['New_Aff_Sigma'].combine_first(t['Aff_Sigma'])
    t['Aff_Rounds'] += t['Inc_Aff_Rounds'].fillna(0).astype(int)
    t.drop(columns=['Aff','New_Mu','New_Sigma','New_Aff_Mu','New_Aff_Sigma','Inc_Aff_Rounds'], inplace=True)

    t = t.merge(
        neg_updates, left_on='Team', right_on='Neg', how='left'
    )
    t['Mu'] = t['New_Mu'].combine_first(t['Mu'])
    t['Sigma'] = t['New_Sigma'].combine_first(t['Sigma'])
    t['Neg_Mu'] = t['New_Neg_Mu'].combine_first(t['Neg_Mu'])
    t['Neg_Sigma'] = t['New_Neg_Sigma'].combine_first(t['Neg_Sigma'])
    t['Neg_Rounds'] += t['Inc_Neg_Rounds'].fillna(0).astype(int)
    t.drop(columns=['Neg','New_Mu','New_Sigma','New_Neg_Mu','New_Neg_Sigma','Inc_Neg_Rounds'], inplace=True)

    t['Conservative'] = t['Mu'] - 3*t['Sigma']
    return t[['Team','Mu','Sigma','Aff_Mu','Aff_Sigma','Neg_Mu','Neg_Sigma','Aff_Rounds','Neg_Rounds','Conservative']]


def trueskill_all(teams: pd.DataFrame, rd: pd.DataFrame, env: TrueSkill | None = None):
    """
    One-shot update for all rounds.
    """
    aff, neg, env, teams = setup(teams, rd, env)
    aff_upd, neg_upd = compute_new_trueskill(aff, neg, env)
    return trueskill_merge(teams, aff_upd, neg_upd)

In [8]:
# ----- load per-round results -----
rds = [
    'nu_1','nu_2','nu_3','nu_4','nu_5','nu_6','nu_7','nu_8',
    'nu_dubs','nu_octas','nu_quarters','nu_semis','nu_finals',
    'uk_1','uk_2','uk_3','uk_4','uk_5','uk_6',
    'uk_dubs','uk_octas','uk_quarters','uk_semis','uk_finals',
    'gonzaga_1','gonzaga_2','gonzaga_3','gonzaga_4','gonzaga_5','gonzaga_6','gonzaga_7','gonzaga_8',
    'gonzaga_dubs','gonzaga_octas','gonzaga_quarters','gonzaga_semis','gonzaga_finals',
    'wake_1','wake_2','wake_3','wake_4','wake_5','wake_6','wake_7',
    'wake_dubs','wake_octas','wake_quarters','wake_semis','wake_finals',
    'gt_1','gt_2','gt_3','gt_4','gt_5','gt_6','gt_7','gt_8',
    'gt_dubs','gt_octas','gt_quarters','gt_semis','gt_finals'
]

for i in rds:
    clean_rd(i)

results = [pd.read_csv(f"data_labor/{name}.csv") for name in rds]

In [10]:
# ----- load teams (TrueSkill columns expected) -----
teams = pd.read_csv("teams_labor_init.csv").copy()

for i in range(5):
    for k in range(len(rds)):
        print(k, rds[k])
        teams = trueskill_all(teams, results[k])

# ----- finalize / export -----
# Conservative rating for leaderboard
teams['Conservative'] = teams['Mu'] - 3*teams['Sigma']

# Round a bit for readability
round_cols = ['Mu','Sigma','Aff_Mu','Aff_Sigma','Neg_Mu','Neg_Sigma', 'Conservative']
teams[round_cols] = teams[round_cols].astype(float).round(3)
teams[['Aff_Rounds', 'Neg_Rounds']] /= 5
teams = teams[(teams['Aff_Rounds'] > 0) | (teams['Neg_Rounds'] > 0)]
teams = teams.sort_values('Mu', ascending=False).reset_index(drop=True)

teams.to_csv("teams_labor.csv", index=False)

0 nu_1
1 nu_2
2 nu_3
3 nu_4
4 nu_5
5 nu_6
6 nu_7
7 nu_8
8 nu_dubs
9 nu_octas
10 nu_quarters
11 nu_semis
12 nu_finals
13 uk_1
14 uk_2
15 uk_3
16 uk_4
17 uk_5
18 uk_6
19 uk_dubs
20 uk_octas
21 uk_quarters
22 uk_semis
23 uk_finals
24 gonzaga_1
25 gonzaga_2
26 gonzaga_3
27 gonzaga_4
28 gonzaga_5
29 gonzaga_6
30 gonzaga_7
31 gonzaga_8
32 gonzaga_dubs
33 gonzaga_octas
34 gonzaga_quarters
35 gonzaga_semis
36 gonzaga_finals
37 wake_1
38 wake_2
39 wake_3
40 wake_4
41 wake_5
42 wake_6
43 wake_7
44 wake_dubs
45 wake_octas
46 wake_quarters
47 wake_semis
48 wake_finals
49 gt_1
50 gt_2
51 gt_3
52 gt_4
53 gt_5
54 gt_6
55 gt_7
56 gt_8
57 gt_dubs
58 gt_octas
59 gt_quarters
60 gt_semis
61 gt_finals
0 nu_1
1 nu_2
2 nu_3
3 nu_4
4 nu_5
5 nu_6
6 nu_7
7 nu_8
8 nu_dubs
9 nu_octas
10 nu_quarters
11 nu_semis
12 nu_finals
13 uk_1
14 uk_2
15 uk_3
16 uk_4
17 uk_5
18 uk_6
19 uk_dubs
20 uk_octas
21 uk_quarters
22 uk_semis
23 uk_finals
24 gonzaga_1
25 gonzaga_2
26 gonzaga_3
27 gonzaga_4
28 gonzaga_5
29 gonzaga_6
30 g

In [22]:
aff = results[40].merge(teams, left_on='Aff', right_on='Team', how='left')
neg = results[40].merge(teams, left_on='Neg', right_on='Team', how='left')

print("Lengths:", len(results[24]), len(aff), len(neg))  # must match
# assert len(results[16]) == len(aff) == len(neg), "Merges misaligned — likely missing teams or bad keys."

# Spot-check missing rating fields (should be 0 if we added missing teams)
print("aff NaNs in rating cols:\n", aff[['Mu','Sigma','Aff_Mu','Aff_Sigma']].isna().sum())
print("neg NaNs in rating cols:\n", neg[['Mu','Sigma','Neg_Mu','Neg_Sigma']].isna().sum())

# Peek at rows where something is missing
bad_aff_idx = aff[aff['Mu'].isna() | aff['Sigma'].isna() | aff['Aff_Mu'].isna() | aff['Aff_Sigma'].isna()].index
bad_neg_idx = neg[neg['Mu'].isna() | neg['Sigma'].isna() | neg['Neg_Mu'].isna() | neg['Neg_Sigma'].isna()].index
print("Problematic aff rows:", bad_aff_idx[:10].tolist())
print("Problematic neg rows:", bad_neg_idx[:10].tolist())

if len(bad_aff_idx) or len(bad_neg_idx):
    display(aff.loc[bad_aff_idx].head(3))
    display(neg.loc[bad_neg_idx].head(3))


Lengths: 41 70 70
aff NaNs in rating cols:
 Mu           0
Sigma        0
Aff_Mu       0
Aff_Sigma    0
dtype: int64
neg NaNs in rating cols:
 Mu           1
Sigma        1
Neg_Mu       1
Neg_Sigma    1
dtype: int64
Problematic aff rows: []
Problematic neg rows: [4]


,Unnamed: 0,Aff,Neg,Win,Team,Mu,Sigma,Aff_Mu,Aff_Sigma,Neg_Mu,Neg_Sigma,Aff_Rounds,Neg_Rounds,Conservative


,Unnamed: 0,Aff,Neg,Win,Team,Mu,Sigma,Aff_Mu,Aff_Sigma,Neg_Mu,Neg_Sigma,Aff_Rounds,Neg_Rounds,Conservative
4,4,Missouri State BC,NaN,Aff,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
